In [ ]:
import pandas as pd
import numpy as np
import requests
import json

In [2]:
teams_25_26 = [
    {"name":"Arsenal", "id":"3","shortName":"Arsenal","abbr":"ARS"},
    {"name":"Aston Villa", "id":"7","shortName":"Aston Villa","abbr":"AVL"},
    {"name":"Bournemouth", "id":"91","shortName":"Bournemouth","abbr":"BOU"},
    {"name":"Brentford", "id":"94","shortName":"Brentford","abbr":"BRE"},
    {"name" :"Brighton and Hove Albion","id":"36","shortName":"Brighton","abbr":"BHA"},
    {"name":"Burnley","id":"90","shortName":"Burnley","abbr":"BUR"},
    {"name":"Chelsea","id":"8","shortName":"Chelsea","abbr":"CHE"},
    {"name":"Crystal Palace","id":"31","shortName":"Crystal Palace","abbr":"CRY"},
    {"name":"Everton","id":"11","shortName":"Everton","abbr":"EVE"},
    {"name":"Fulham","id":"54","shortName":"Fulham","abbr":"FUL"},
    {"name":"Leeds United","id":"2","shortName":"Leeds","abbr":"LEE"},
    {"name":"Liverpool","id":"14","shortName":"Liverpool","abbr":"LIV"},
    {"name":"Manchester City","id":"43","shortName":"Man City","abbr":"MCI"},
    {"name":"Manchester United","id":"1","shortName":"Man Utd","abbr":"MUN"},
    {"name":"Newcastle United","id":"4","shortName":"Newcastle","abbr":"NEW"},
    {"name":"Nottingham Forest","id":"17","shortName":"Nott'm Forest","abbr":"NFO"},
    {"name":"Sunderland","id":"56","shortName":"Sunderland","abbr":"SUN"},
    {"name":"Tottenham Hotspur","id":"6","shortName":"Spurs","abbr":"TOT"},
    {"name":"West Ham United","id":"21","shortName":"West Ham","abbr":"WHU"},
    {"name":"Wolverhampton Wanderers","id":"39","shortName":"Wolves","abbr":"WOL"}]

team_names =   {team['shortName']: team['name'] for team in teams_25_26}
# create lookup dictionary
team_name_to_id_25_26 = {team['shortName']: team['name'] for team in teams_25_26}

player_dict_ids = pd.read_csv('./data/vaastav/data/id_dict_25_26.csv')

player_dict_ids.rename(columns={'FPL_ID': 'fpl_id','FPL_Name': 'fpl_name' , 'Understat_ID': 'understat_id', 'Understat_Name': 'understat_name'}, inplace=True)
player_dict_ids

,fpl_id,fpl_name,understat_id,understat_name
0,116,Aaron Hickey,8942,Aaron Hickey
3,610,Aaron Wan-Bissaka,5584,Aaron Wan-Bissaka
7,406,Abdukodir Khusanov,11763,Abduqodir Khusanov
8,326,Adama Traoré Diarra,900,Adama Traoré
10,73,Adam Smith,825,Adam Smith
...,...,...,...,...
724,48,Youri Tielemans,5956,Youri Tielemans
728,712,Yéremy Pino Santos,9024,Yeremi Pino
734,215,Zian Flemming,13729,Zian Flemming
736,713,Álex Jiménez Sánchez,12168,Alejandro Jiménez


In [ ]:
# Get FPL data from the official API
url = "https://fantasy.premierleague.com/api/bootstrap-static/"
response = requests.get(url)
data = response.json()

# Extract teams and players
teams = {team["id"]: team["name"] for team in data["teams"]}
players = data["elements"]

# Function to get players from a specific team
def get_players_for_team(team_name):
    team_id = next((id for id, name in teams.items() if name.lower() == team_name.lower()), None)
    # print(team_id)
    if team_id is None:
        return f"Team '{team_name}' not found."

    team_players = [player for player in players if player["team"] == team_id]

    # print(team_players)
    return [
        {'fpl_id': player['id'] ,"fpl_name": f"{player['first_name']} {player['second_name']}", "fpl_position": player["element_type"], 'team': team_name}
        for player in team_players
    ]

# Example: Get players from Manchester City
# team_name = 'Man City'
players_list = [] # get_players_for_team(team_name)

for team in list(teams.values()):
    players_list.extend(get_players_for_team(team))


for player in players_list:
        player['full_team_name'] = team_names[player['team']]

fpl_data = pd.DataFrame(players_list)
fpl_data

,fpl_id,fpl_name,fpl_position,team,full_team_name
0,1,David Raya Martín,1,Arsenal,Arsenal
1,2,Kepa Arrizabalaga Revuelta,1,Arsenal,Arsenal
2,3,Karl Hein,1,Arsenal,Arsenal
3,4,Tommy Setford,1,Arsenal,Arsenal
4,5,Gabriel dos Santos Magalhães,2,Arsenal,Arsenal
...,...,...,...,...,...
736,663,Jhon Arias,3,Wolves,Wolverhampton Wanderers
737,682,David Møller Wolfe,2,Wolves,Wolverhampton Wanderers
738,695,Jackson Tchatchoua,2,Wolves,Wolverhampton Wanderers
739,709,Ladislav Krejcí,2,Wolves,Wolverhampton Wanderers


In [7]:
# Combine player_dict_ids with fpl_data on fpl_id
combined_df = pd.merge(fpl_data[['fpl_id', 'fpl_position', 'team', 'full_team_name']], player_dict_ids, on='fpl_id', how='left')

final_df = combined_df[~combined_df['understat_id'].isnull()]
final_df.to_csv('./fpl_understat_id_name.csv', index=False)

In [8]:
final_df

,fpl_id,fpl_position,team,full_team_name,fpl_name,understat_id,understat_name
0,1,1,Arsenal,Arsenal,David Raya Martín,9676.0,David Raya
4,5,2,Arsenal,Arsenal,Gabriel dos Santos Magalhães,5613.0,Gabriel
5,6,2,Arsenal,Arsenal,William Saliba,6888.0,William Saliba
6,7,2,Arsenal,Arsenal,Riccardo Calafiori,8129.0,Riccardo Calafiori
7,8,2,Arsenal,Arsenal,Jurriën Timber,11707.0,Jurriën Timber
...,...,...,...,...,...,...,...
736,663,3,Wolves,Wolverhampton Wanderers,Jhon Arias,13741.0,Jhon Arias
737,682,2,Wolves,Wolverhampton Wanderers,David Møller Wolfe,13740.0,David Møller Wolfe
738,695,2,Wolves,Wolverhampton Wanderers,Jackson Tchatchoua,12113.0,Jackson Tchatchoua
739,709,2,Wolves,Wolverhampton Wanderers,Ladislav Krejcí,12721.0,Ladislav Krejcí
